# Project 23 — Production AI Platform (gateway → LLM/VLM → observability)

```
mock LLM/VLM backends (rule-based, fixed sleeps: honest latency stubs, no weights)
  -> gateway pipeline: auth -> ratelimit -> admission(queue) -> cache -> router -> backend
  -> observability: per-route counters + latencies (/metrics), trace ids (/trace log), /gpu stub
  -> TestClient only (NO ports, NO network) -> contracts: 401/429/503 shapes, cache hits, batch gain
  -> load ledger: scripted + bursty traffic; artifacts: platform.csv + summary.csv + platform.png
```

Goal: learn which platform layer rescues which failure (auth→abuse, limit→burst, queue→overload,
cache→repeats, batch→throughput, metrics/trace→debugging) and meter each one.
Objectives: state what each of auth/ratelimit/queue/batch/cache/metrics/trace/gpu-monitor answers.


In [ ]:
import os # need os to make results/
import random # need random to seed key jitter in the ledger
import time # need time for sleep-based latency stubs + wall clocks
import csv # need csv to write platform.csv + summary.csv
import threading # need threads for the burst-overload probe
import concurrent.futures as cf # need a pool to fire bursty requests
import numpy as np # need numpy for latency means/percentiles
import matplotlib # need matplotlib for the headless figure
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for latency + status figure
from fastapi import FastAPI, Request # need FastAPI + Request for routes + headers
from fastapi.responses import JSONResponse # need JSONResponse for shaped error bodies
from fastapi.testclient import TestClient # need TestClient: in-process HTTP, no ports
SEED = 23 # fixed seed: same ledger on every rerun
random.seed(SEED) # seed stdlib RNG
np.random.seed(SEED) # seed numpy RNG
os.makedirs('results', exist_ok=True) # make results/ for CSVs + PNG
TOKEN = 'secret-token' # the one valid bearer token (everything else -> 401)
RL_MAX = 5 # rate budget: 5 admitted requests per key per window
MAX_INFLIGHT = 4 # admission bound: >4 concurrent -> 429 (queue full)
CACHE = {} # exact-prompt cache: prompt -> answer (repeat traffic is free)
USED = {} # rate-limit ledger: key -> admitted count this window
INFLIGHT = threading.Semaphore(MAX_INFLIGHT) # admission gate (non-blocking acquire)
COUNTS = {} # metrics counter: (route, status) -> n
LATS = {} # metrics latencies: route -> [ms]
TRACE = [] # trace log: one span dict per request (id, route, status, ms)
TID = [0] # mutable trace-id counter (list so helpers can bump it)
GPU_SIM = {'cuda': False, 'simulated': True, 'util_pct': 37.5, 'mem_MB': 2048} # stubbed monitor (no GPU here)
def note(route, status, ms, tid): # one observability write per request (metrics + trace)
    COUNTS[(route, status)] = COUNTS.get((route, status), 0) + 1 # bump the counter
    LATS.setdefault(route, []).append(ms) # append the latency sample
    TRACE.append({'id': tid, 'route': route, 'status': status, 'ms': round(ms, 3)}) # append the span
def reset_state(): # reset mutable platform state between test phases
    CACHE.clear() # empty the prompt cache
    USED.clear() # empty the rate-limit ledger
    COUNTS.clear() # empty the metrics counters
    LATS.clear() # empty the latency samples
    TRACE.clear() # empty the trace log
    TID[0] = 0 # restart trace ids at 1
print('gateway constants:', TOKEN[:6] + '...', RL_MAX, MAX_INFLIGHT) # confirm the policy knobs

In [ ]:
# ---- backends: deterministic stubs with fixed costs (stand-ins for real LLM/VLM servers) ----
def llm_backend(prompt, hold=0.0): # rule-based LLM: reverse-words "generation" + 8ms prefill cost
    time.sleep(0.008 + hold) # fixed 8ms prefill stub (+ optional hold for overload tests)
    words = prompt.split() # tokenize on whitespace (stub-grade)
    return ' '.join(words[::-1]) + f' [{len(words)}tok]' # reversed words + token count (deterministic)
def vlm_backend(tag, hold=0.0): # rule-based VLM: tag -> caption table + 5ms vision cost
    time.sleep(0.005 + hold) # fixed 5ms vision stub
    table = {'pitch': 'aerial pitch, 4 players, ball near center', # known tag 1
             'chart': 'bar chart, bar2 tallest', # known tag 2
             'ocr': 'jersey number 10'} # known tag 3
    return table.get(tag, 'unknown scene') # closed vocabulary (unknown -> fallback string)
# ---- app: gateway pipeline auth -> ratelimit -> admission -> cache -> router -> backend ----
app = FastAPI(title='prod-ai-platform (P23)') # the ASGI app TestClient will drive
def gate(request, route): # auth + ratelimit + admission, in gateway order; returns (err_response, key)
    auth = request.headers.get('authorization', '') # read the bearer header
    if auth != f'Bearer {TOKEN}': # wrong or missing token
        return JSONResponse({'error': 'unauthorized'}, status_code=401), None # 401 before any work
    key = request.headers.get('x-key', 'default') # per-client key for rate limiting
    if USED.get(key, 0) >= RL_MAX: # budget exhausted for this key
        return JSONResponse({'error': 'rate-limited'}, status_code=429), None # 429: slow down, not broken
    if not INFLIGHT.acquire(blocking=False): # all 4 slots busy
        return JSONResponse({'error': 'queue-full'}, status_code=429), None # 429: shed load, never 500
    USED[key] = USED.get(key, 0) + 1 # admit: charge the rate budget
    return None, key # admitted: caller must release INFLIGHT
@app.get('/health') # liveness probe (no auth: load balancers call this)
def health(): # health handler
    return {'ok': True} # plain 200, never gated
@app.post('/generate') # LLM route: single prompt
def generate(body: dict, request: Request): # body has prompt (+ optional hold_ms for tests)
    t0 = time.perf_counter() # start the wall clock
    TID[0] += 1 # mint a trace id
    err, _ = gate(request, 'llm') # run the three gates
    if err is not None: # rejected at the gateway
        note('llm', err.status_code, (time.perf_counter() - t0) * 1e3, TID[0]) # meter the rejection too
        return err # shaped 401/429 body
    try: # admitted: run the backend
        p = body.get('prompt', '') # the prompt string
        if p in CACHE: # exact repeat?
            ans, hit = CACHE[p], True # cache hit: zero backend cost
        else: # cache miss
            ans = llm_backend(p, body.get('hold_ms', 0) / 1000.0) # run stub (optional hold for burst test)
            CACHE[p] = ans # populate the cache
            hit = False # mark the miss
        ms = (time.perf_counter() - t0) * 1e3 # wall ms including stub sleep
        note('llm', 200, ms, TID[0]) # meter the success
        return {'answer': ans, 'cached': hit, 'trace_id': TID[0]} # answer + observability bits
    finally: # always release the slot
        INFLIGHT.release() # free one admission slot
@app.post('/batch') # batch route: k prompts share one 8ms prefill (+1ms each)
def batch(body: dict, request: Request): # body has prompts: [...]
    t0 = time.perf_counter() # start the wall clock
    TID[0] += 1 # mint a trace id
    err, _ = gate(request, 'batch') # same three gates
    if err is not None: # rejected
        note('batch', err.status_code, (time.perf_counter() - t0) * 1e3, TID[0]) # meter rejection
        return err # shaped 401/429
    try: # admitted
        ps = body.get('prompts', []) # the prompt list
        time.sleep(0.008 + 0.001 * len(ps)) # shared prefill: 8ms once + 1ms per item (batching dividend)
        ans = [p.split()[::-1] for p in ps] # stub "generation" per item (reversed words, no per-item sleep)
        ans = [' '.join(a) for a in ans] # join back to strings
        ms = (time.perf_counter() - t0) * 1e3 # wall ms for the whole batch
        note('batch', 200, ms, TID[0]) # meter the success
        return {'answers': ans, 'trace_id': TID[0], 'ms': round(ms, 3)} # answers + timing proof
    finally: # always release
        INFLIGHT.release() # free the slot
@app.post('/caption') # VLM route: image tag -> caption (router picks backend by path)
def caption(body: dict, request: Request): # body has tag (+ optional hold_ms)
    t0 = time.perf_counter() # start the wall clock
    TID[0] += 1 # mint a trace id
    err, _ = gate(request, 'vlm') # same three gates
    if err is not None: # rejected
        note('vlm', err.status_code, (time.perf_counter() - t0) * 1e3, TID[0]) # meter rejection
        return err # shaped 401/429
    try: # admitted
        ans = vlm_backend(body.get('tag', ''), body.get('hold_ms', 0) / 1000.0) # run the vision stub
        ms = (time.perf_counter() - t0) * 1e3 # wall ms
        note('vlm', 200, ms, TID[0]) # meter success
        return {'caption': ans, 'trace_id': TID[0]} # caption + trace id
    finally: # always release
        INFLIGHT.release() # free the slot
@app.get('/metrics') # observability: counters + mean latency per route (auth-gated like a real admin port)
def metrics(request: Request): # metrics handler
    if request.headers.get('authorization', '') != f'Bearer {TOKEN}': # same auth bar
        return JSONResponse({'error': 'unauthorized'}, status_code=401) # 401 without token
    return {'counts': {f'{r}:{s}': n for (r, s), n in COUNTS.items()}, # flattened counter map
            'mean_ms': {r: round(float(np.mean(v)), 3) for r, v in LATS.items()}, # mean latency per route
            'cache_size': len(CACHE)} # cache occupancy
@app.get('/gpu') # gpu monitor stub: real fields when CUDA exists, labeled simulation otherwise
def gpu(request: Request): # monitor handler
    if request.headers.get('authorization', '') != f'Bearer {TOKEN}': # same auth bar
        return JSONResponse({'error': 'unauthorized'}, status_code=401) # 401 without token
    try: # try a real reading first
        import torch # torch is installed in this repo
        if torch.cuda.is_available(): # a real GPU is present
            return {'cuda': True, 'simulated': False, # real reading follows
                    'devices': torch.cuda.device_count()} # device count from torch
    except Exception: # torch missing or CUDA call failed
        pass # fall through to the stub
    return GPU_SIM # labeled simulation (honest on CPU machines)
print('routes: /health /generate /batch /caption /metrics /gpu') # confirm the surface

In [ ]:
# ---- contracts: every platform promise as an assertion (must all pass before load tests) ----
cli = TestClient(app) # in-process client: no ports, no network, deterministic
H = {'Authorization': f'Bearer {TOKEN}'} # valid auth header used below
r = cli.get('/health') # liveness probe (unauthenticated by design)
assert r.status_code == 200 and r.json()['ok'] is True # health contract
r = cli.post('/generate', json={'prompt': 'x'}, headers={}) # no token at all
assert r.status_code == 401 # auth contract: missing token -> 401
r = cli.post('/generate', json={'prompt': 'x'}, headers={'Authorization': 'Bearer wrong'}) # wrong token
assert r.status_code == 401 # auth contract: wrong token -> 401
reset_state() # clean slate for the cache contract
r1 = cli.post('/generate', json={'prompt': 'the ball nears goal'}, headers=H) # first sight: miss
r2 = cli.post('/generate', json={'prompt': 'the ball nears goal'}, headers=H) # exact repeat: hit
assert r1.json()['cached'] is False and r2.json()['cached'] is True # cache contract
assert r1.json()['answer'] == r2.json()['answer'] # cache consistency contract
reset_state() # clean slate for the batch contract
s = cli.post('/generate', json={'prompt': 'one two three four'}, headers=H) # single baseline
b = cli.post('/batch', json={'prompts': ['one two three four'] * 4}, headers=H) # batch of 4 same-size
single_ms = LATS['llm'][-1] # measured single latency (last llm sample)
batch_ms = LATS['batch'][-1] # measured batch latency (whole batch of 4)
assert batch_ms < 4 * single_ms # batching contract: sharing the 8ms prefill wins
reset_state() # clean slate for the rate-limit contract
codes = [cli.post('/generate', json={'prompt': f'q{i}'}, headers={**H, 'x-key': 'u1'}).status_code # 7 sends, budget 5
         for i in range(7)] # one key, seven requests
assert codes.count(429) == 2 and codes.count(200) == 5 # rate-limit contract: 5 in, 2 shed
reset_state() # clean slate for the router contract
c = cli.post('/caption', json={'tag': 'pitch'}, headers=H) # vision route, known tag
assert c.status_code == 200 and 'pitch' in c.json()['caption'] # router contract: tag -> pitch caption
g = cli.get('/gpu', headers=H) # monitor route
assert 'cuda' in g.json() and 'simulated' in g.json() # monitor contract: labeled reading
m = cli.get('/metrics', headers=H) # observability route
assert 'counts' in m.json() and 'mean_ms' in m.json() # metrics contract: counters + latencies present
print('contracts pass: 401/429/cache/batch/router/metrics/gpu') # all green
print('single_ms', round(single_ms, 2), 'batch4_ms', round(batch_ms, 2)) # the batching dividend, measured

In [ ]:
# ---- load ledger: scripted mix + burst overload; every request becomes one CSV row ----
reset_state() # clean slate for the ledger phase
ledger = [] # platform.csv rows
def fire(route, body, key='ledger', auth=True): # one ledgered request: send + record the outcome
    h = dict(H) # start from valid auth
    h['x-key'] = key # per-script key (isolates phases from each other)
    if not auth: # unauthenticated variant
        h.pop('Authorization') # drop the token -> expect 401
    t0 = time.perf_counter() # client-side wall clock
    if route == 'llm': # LLM route
        r = cli.post('/generate', json=body, headers=h) # send single
    elif route == 'batch': # batch route
        r = cli.post('/batch', json=body, headers=h) # send batch
    else: # VLM route
        r = cli.post('/caption', json=body, headers=h) # send caption
    ms = (time.perf_counter() - t0) * 1e3 # client-measured ms
    j = r.json() # parse the body
    ledger.append({'phase': 'mix', 'route': route, 'status': r.status_code, # outcome cols
                   'cached': j.get('cached', ''), 'ms': round(ms, 3), # cache flag + latency
                   'trace_id': j.get('trace_id', '')}) # trace id when admitted
    return r # return for inline asserts
for i in range(4): # 4 fresh LLM prompts (misses)
    fire('llm', {'prompt': f'match event number {i}'}, key='k-mix') # distinct prompts
fire('llm', {'prompt': 'match event number 1'}, key='k-mix') # 1 repeat (expect cached=true)
fire('batch', {'prompts': ['a b c', 'd e f', 'g h i']}, key='k-batch') # 1 batch of 3
fire('vlm', {'tag': 'chart'}, key='k-vlm') # 1 vision call
fire('llm', {'prompt': 'no auth here'}, key='k-mix', auth=False) # 1 unauthenticated (expect 401)
for i in range(6): # 6 sends on a 5-budget key (last one sheds)
    fire('llm', {'prompt': f'rl probe {i}'}, key='k-tight') # budget probe
# ---- burst: 8 threads x 300ms-hold against 4 slots -> 200s + 429s, zero 500s ----
reset_state() # fresh budgets so the burst measures admission, not rate limits
import threading as _th # local alias for the barrier (keeps cell-1 list canonical)
bar = _th.Barrier(8) # align all 8 threads for maximum overlap
def burst_hit(i): # one bursty request: wait at barrier, then hold a slot 300ms
    bar.wait() # align starts (worst-case overlap)
    t0 = time.perf_counter() # client clock
    r = cli.post('/generate', json={'prompt': f'burst {i}', 'hold_ms': 300}, # 300ms hold pins the slot
                 headers={**H, 'x-key': f'b{i}'}) # unique key per thread (rate limit never trips)
    ms = (time.perf_counter() - t0) * 1e3 # client ms
    ledger.append({'phase': 'burst', 'route': 'llm', 'status': r.status_code, # outcome cols
                   'cached': '', 'ms': round(ms, 3), 'trace_id': ''}) # burst rows (no trace needed)
    return r.status_code # status for the tally
with cf.ThreadPoolExecutor(max_workers=8) as ex: # 8 threads vs 4 slots
    codes = list(ex.map(burst_hit, range(8))) # fire all 8, collect statuses
assert 500 not in codes # overload contract: shed, never crash
assert codes.count(429) >= 1 # overload contract: at least one shed (usually 4)
assert codes.count(200) >= 1 # liveness contract: someone always gets through
print('burst codes:', sorted(codes)) # the shed pattern, measured
print('ledger rows:', len(ledger)) # mix + burst rows total

In [ ]:
# ---- summarize: per-phase table + CSVs + figure ----
mix = [l for l in ledger if l['phase'] == 'mix'] # scripted-mix rows
bst = [l for l in ledger if l['phase'] == 'burst'] # burst rows
def summ(rows): # summarize one phase: n, 200/401/429 shares, mean ms
    n = len(rows) # row count
    ok = sum(1 for r in rows if r['status'] == 200) # successes
    r429 = sum(1 for r in rows if r['status'] == 429) # shed
    r401 = sum(1 for r in rows if r['status'] == 401) # unauthenticated
    return {'n': n, 'ok': ok, 'r429': r429, 'r401': r401, # count cols
            'mean_ms': round(float(np.mean([r['ms'] for r in rows])), 2)} # latency col
srows = [{'phase': 'mix', **summ(mix)}, {'phase': 'burst', **summ(bst)}] # two summary rows
for s in srows: # echo to the log
    print(s) # one row per phase
assert srows[0]['r401'] == 1 # mix must contain exactly the one unauthenticated probe
assert srows[0]['r429'] == 1 # mix must contain exactly the one tight-budget shed
assert srows[1]['r429'] >= 1 and srows[1]['n'] == 8 # burst: 8 sends, some shed
with open('results/platform.csv', 'w', newline='') as f: # write the per-request ledger
    w = csv.DictWriter(f, fieldnames=list(ledger[0].keys())) # header from row keys
    w.writeheader() # write header
    w.writerows(ledger) # write all rows
with open('results/summary.csv', 'w', newline='') as f: # write the per-phase summary
    w = csv.DictWriter(f, fieldnames=list(srows[0].keys())) # header from row keys
    w.writeheader() # write header
    w.writerows(srows) # write 2 rows
fig, ax = plt.subplots(1, 2, figsize=(11, 4)) # two panels: latency + status mix
routes = ['llm', 'batch', 'vlm'] # the three serving routes
means = [float(np.mean(LATS[r])) if r in LATS and LATS[r] else 0.0 for r in routes] # mean ms per route
ax[0].bar(routes, means) # latency bars
ax[0].set_ylabel('mean ms (server-measured)') # y label
ax[0].set_title('Latency by route (post-reset burst state)') # panel title
labels = ['200', '429', '401'] # status classes
vals = [sum(1 for l in ledger if l['status'] == int(s)) for s in labels] # counts per class
ax[1].bar(labels, vals) # status bars
ax[1].set_ylabel('requests') # y label
ax[1].set_title('Ledger outcomes (mix + burst)') # panel title
plt.tight_layout() # avoid clipping
plt.savefig('results/platform.png', dpi=100) # save the figure
print('wrote results/platform.csv results/summary.csv results/platform.png') # artifact confirmation
print('cache entries:', len(CACHE), '| trace spans:', len(TRACE)) # observability footprint

Cleanup / next steps:
- Artifacts: `results/platform.csv` (per-request ledger), `results/summary.csv` (mix/burst phases),
  `results/platform.png` (latency-by-route + outcome bars).
- To go real: swap `llm_backend`/`vlm_backend` for vLLM endpoints (P18 scheduler behind `/batch`),
  Redis for `CACHE`, a token-bucket sidecar for `USED`, OTel spans for `TRACE`, DCGM for `/gpu`.
- Next: P24 (capstone) wires P8 detection + P9 tracking + P19 vision + P20 retrieval + this gateway
  into one sports-AI platform: video → tracks → stats → report → API.
